# Heart Disease Prediction — Model Training
Trains multiple classifiers, picks the best by cross-validated F1, saves artifact.

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

# Classifiers to compare
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('heart.csv')
X = df.drop('target', axis=1)
y = df['target']
feature_names = list(X.columns)
print('Shape:', X.shape, '| Classes:', y.value_counts().to_dict())

# ── Train / test split (stratified) ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Scale ────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Candidate models ─────────────────────────────────────────────────────────
candidates = {
    'Gradient Boosting':  GradientBoostingClassifier(n_estimators=200, learning_rate=0.08,
                                                      max_depth=3, random_state=42),
    'Random Forest':      RandomForestClassifier(n_estimators=200, max_depth=6,
                                                  random_state=42, class_weight='balanced'),
    'Logistic Regression':LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'KNN (k=7)':          KNeighborsClassifier(n_neighbors=7, metric='minkowski'),
    'SVM (RBF)':          SVC(probability=True, C=1.0, kernel='rbf', random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('\n── Cross-validation (5-fold, F1) ─────────────────────────')
for name, clf in candidates.items():
    scores = cross_val_score(clf, X_train_s, y_train, cv=cv, scoring='f1')
    results[name] = scores.mean()
    print(f'  {name:25s}  F1 = {scores.mean():.4f} ± {scores.std():.4f}')

# ── Pick winner ──────────────────────────────────────────────────────────────
best_name = max(results, key=results.get)
print(f'\n✅  Best model: {best_name}  (CV F1 = {results[best_name]:.4f})')

# ── Final fit on full train set ───────────────────────────────────────────────
model = candidates[best_name]
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print('\n── Test-set metrics ──────────────────────────────────────')
print(f'  Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'  ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

# ── Save artifact ────────────────────────────────────────────────────────────
Path('artifacts').mkdir(exist_ok=True)
artifact = {
    'model':         model,
    'scaler':        scaler,
    'feature_names': feature_names,
    'model_name':    best_name,
}
with open('artifacts/heart_disease_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print('\n✅  Artifact saved → artifacts/heart_disease_model.pkl')

# ── Quick sanity check (same sample as before) ───────────────────────────────
sample = np.array([[63,1,3,145,233,1,0,150,0,2.3,0,0,1]])
sample_s = scaler.transform(sample)
print('\nSanity check (sample=[63,1,3,145,233,1,0,150,0,2.3,0,0,1]):')
print('  Prediction:', model.predict(sample_s))
print('  Probability:', model.predict_proba(sample_s).round(3))


Shape: (303, 13) | Classes: {1: 165, 0: 138}

── Cross-validation (5-fold, F1) ─────────────────────────
  Gradient Boosting          F1 = 0.7764 ± 0.0821
  Random Forest              F1 = 0.8406 ± 0.0520
  Logistic Regression        F1 = 0.8341 ± 0.0948
  KNN (k=7)                  F1 = 0.8358 ± 0.0441
  SVM (RBF)                  F1 = 0.8179 ± 0.0529

✅  Best model: Random Forest  (CV F1 = 0.8406)

── Test-set metrics ──────────────────────────────────────
  Accuracy : 0.8197
  ROC-AUC  : 0.9102

              precision    recall  f1-score   support

  No Disease       0.90      0.68      0.78        28
     Disease       0.78      0.94      0.85        33

    accuracy                           0.82        61
   macro avg       0.84      0.81      0.81        61
weighted avg       0.83      0.82      0.82        61


✅  Artifact saved → artifacts/heart_disease_model.pkl

Sanity check (sample=[63,1,3,145,233,1,0,150,0,2.3,0,0,1]):
  Prediction: [1]
  Probability: [[0.325 0.675]]


d:\xode-2\internship-glowlogics\Projects\heart-disease-prediction\env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
